# NB11 - Mixed Precision & Gradient Scaling
## Background
Training large models in FP32 requires 4 bytes per parameter. FP16 reduces this to 2 bytes, enabling 2x larger batch sizes and faster matrix multiplications on modern GPUs (tensor cores). However, FP16 has a limited dynamic range: values below ~6e-5 underflow to zero, and gradients in early training are often this small. Loss scaling (Micikevicius et al., 2018) multiplies the loss by a large scalar before backward, shifting gradients into FP16's representable range.

BF16 (Brain Float 16) has the same exponent range as FP32 but lower mantissa precision, avoiding the underflow problem and making it the preferred format for LLM training on TPUs and modern NVIDIA GPUs (Ampere+). Gradient checkpointing trades compute for memory by recomputing activations during backward instead of storing them.

## What You'll Learn
- FP16 vs BF16 vs FP32: numeric ranges and precision comparison
- Loss scaling: dynamic scale factor with overflow detection
- Gradient checkpointing: memory vs compute tradeoff calculation
- Mixed precision training pipeline simulation

## Why This Matters
Mixed precision is non-negotiable for training models above 1B parameters. Understanding loss scaling prevents mysterious NaN losses and explains why overflow detection is needed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import struct

# ── Float format analysis ─────────────────────────────────────────────────
formats = {
    'FP32':  {'bits': 32, 'exp_bits': 8,  'mantissa_bits': 23, 'max': 3.4e38,  'min_normal': 1.2e-38},
    'FP16':  {'bits': 16, 'exp_bits': 5,  'mantissa_bits': 10, 'max': 65504,   'min_normal': 6.1e-5},
    'BF16':  {'bits': 16, 'exp_bits': 8,  'mantissa_bits': 7,  'max': 3.4e38,  'min_normal': 1.2e-38},
    'FP8 E4M3': {'bits': 8, 'exp_bits': 4, 'mantissa_bits': 3, 'max': 448,    'min_normal': 1.56e-2},
}

print(f'{'Format':12s} | {'Bits':5s} | {'Exp bits':9s} | {'Mant bits':10s} | {'Max value':12s} | {'Min normal':12s}')
print('-' * 70)
for name, fmt in formats.items():
    print(f"{name:12s} | {fmt['bits']:5d} | {fmt['exp_bits']:9d} | {fmt['mantissa_bits']:10d} | {fmt['max']:12.2e} | {fmt['min_normal']:12.2e}")

print()
print('FP16 problem: gradients < 6.1e-5 underflow to ZERO')
print('  Typical early-training gradients: 1e-6 to 1e-4 -> underflow risk!')
print()
print('BF16 solution: same exponent range as FP32, no underflow for normal training')
print('  Trade-off: 7-bit mantissa vs 23-bit FP32 -> lower precision, not a problem for SGD')

In [ ]:
# ── Loss scaling simulation ────────────────────────────────────────────────
class DynamicLossScaler:
    """Simulates PyTorch's GradScaler behavior."""
    def __init__(self, init_scale: float = 2**16,
                 growth_factor: float = 2.0,
                 backoff_factor: float = 0.5,
                 growth_interval: int = 2000):
        self.scale = init_scale
        self.growth_factor = growth_factor
        self.backoff_factor = backoff_factor
        self.growth_interval = growth_interval
        self._steps_since_update = 0
        self.history = []

    def scale_loss(self, loss: float) -> float:
        return loss * self.scale

    def step(self, grads_have_overflow: bool) -> bool:
        """Returns True if optimizer step was taken (no overflow)."""
        self.history.append(self.scale)
        if grads_have_overflow:
            # Backoff: reduce scale to avoid repeat overflow
            self.scale *= self.backoff_factor
            self._steps_since_update = 0
            return False  # Skip optimizer step
        else:
            self._steps_since_update += 1
            if self._steps_since_update >= self.growth_interval:
                self.scale *= self.growth_factor
                self._steps_since_update = 0
            return True  # Optimizer step taken

def check_overflow(grads: np.ndarray) -> bool:
    """Check if any gradient is inf or nan (FP16 overflow)."""
    return np.any(~np.isfinite(grads))

# Simulate training with occasional overflow events
np.random.seed(42)
scaler = DynamicLossScaler(init_scale=2**8)
n_steps = 1000

overflow_steps = set(np.random.choice(n_steps, 15, replace=False))
optimizer_steps_taken = 0

for step in range(n_steps):
    has_overflow = step in overflow_steps
    took_step = scaler.step(has_overflow)
    if took_step:
        optimizer_steps_taken += 1

print(f'Total training steps: {n_steps}')
print(f'Overflow events: {len(overflow_steps)}')
print(f'Optimizer steps taken: {optimizer_steps_taken}')
print(f'Final scale factor: {scaler.scale:.0f}')

fig, ax = plt.subplots(figsize=(12, 5))
ax.semilogy(scaler.history, linewidth=1.5, label='Loss scale factor')
for s in sorted(overflow_steps):
    ax.axvline(s, color='red', alpha=0.3, linewidth=0.8)
ax.axvline(sorted(overflow_steps)[0], color='red', alpha=0.5, label='Overflow event')
ax.set_title('Dynamic Loss Scale Factor Over Training')
ax.set_xlabel('Training step'); ax.set_ylabel('Loss scale (log)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{BASE}/s30_11_loss_scaling.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# ── Gradient checkpointing: memory analysis ────────────────────────────────
print('=== Gradient Checkpointing Memory Analysis ===')
print()

# Model parameters: GPT-2 scale
seq_len = 1024
batch_size = 16
d_model = 768
n_layers = 12
n_heads = 12
ffn_mult = 4

bytes_per_elem = 2  # FP16
gb = 1024**3

# Activation memory per layer (simplified)
# Attention: Q, K, V, attention weights, output
attn_activations = batch_size * seq_len * d_model * 5  # Q,K,V,scores,out
# FFN: input, hidden, output
ffn_activations = batch_size * seq_len * d_model * (1 + ffn_mult + 1)
per_layer = (attn_activations + ffn_activations) * bytes_per_elem

# Without checkpointing: store ALL activations
total_no_ckpt = per_layer * n_layers
# With checkpointing: store only sqrt(L) layer boundaries (optimal)
n_checkpoints = int(np.sqrt(n_layers))
total_with_ckpt = per_layer * n_checkpoints
extra_compute = n_layers / n_checkpoints  # recompute ratio

print(f'Model: GPT-2 ({d_model}d, {n_layers}L, seq={seq_len}, batch={batch_size})')
print(f'Activation memory per layer: {per_layer/1e6:.1f} MB')
print()
print(f'Without gradient checkpointing:')
print(f'  Total activation memory: {total_no_ckpt/gb:.2f} GB')
print()
print(f'With gradient checkpointing (sqrt({n_layers})={n_checkpoints} checkpoints):')
print(f'  Total activation memory: {total_with_ckpt/gb:.2f} GB')
print(f'  Memory reduction: {total_no_ckpt/total_with_ckpt:.1f}x')
print(f'  Extra compute: {extra_compute:.1f}x (recompute non-checkpoint layers)')
print()
print('Tradeoff: ~3x memory reduction, ~33% extra compute')
print('At 70B scale, this enables 2x larger batch sizes or longer sequences.')